# PyMIF notebook 09 - legacy options and troubleshooting

This notebook explains the old options that can be confusing when reading older PyMIF examples.

The safe beginner defaults are:

- Use `ZarrManager`, not `ZarrV04Manager`, unless you specifically need the v0.4 compatibility wrapper.
- Use NGFF v0.5 with Zarr v3 for new datasets unless another tool requires v0.4.
- Always make `axes` explicit.
- Always make label metadata explicit with `data_type="label"`.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

In [ ]:
def make_small_raw_manager(seed=0, num_levels=2):
    """Create a tiny TCZYX intensity dataset for examples."""
    rng = np.random.default_rng(seed)
    raw = rng.integers(0, 1200, size=(2, 3, 4, 64, 64), dtype=np.uint16)
    metadata = {
        "name": "small_raw",
        "axes": "tczyx",
        "size": [raw.shape],
        "chunksize": [(1, 1, 2, 32, 32)],
        "scales": [(2.0, 0.30, 0.30)],   # z, y, x spacing because axes contains zyx
        "units": ("micrometer", "micrometer", "micrometer"),
        "time_increment": 60.0,
        "time_increment_unit": "second",
        "channel_names": ["DAPI", "GFP", "RFP"],
        "channel_colors": ["0000FF", "00FF00", "FF0000"],
        "dtype": "uint16",
        "data_type": "intensity",
    }
    manager = mm.ArrayManager(raw, metadata, chunks=metadata["chunksize"][0])
    if num_levels > 1:
        manager.build_pyramid(num_levels=num_levels, downscale_factor=(1, 2, 2))
    return manager

In [ ]:
def label_metadata_from_image_metadata(image_metadata, name="nuclei", dtype="uint32"):
    """Make safe label metadata from image metadata by removing the channel axis.

    Most segmentation labels are one integer label image per time point and do
    not have an image channel axis. This helper avoids confusion from legacy
    TCZYX examples by creating explicit TZYX label metadata.
    """
    md = dict(image_metadata)
    axes = str(md["axes"]).lower()
    if "c" in axes:
        c_axis = axes.index("c")
        md["axes"] = "".join(ax for ax in axes if ax != "c")
        md["size"] = [tuple(v for i, v in enumerate(size) if i != c_axis) for size in md["size"]]
        md["chunksize"] = [tuple(v for i, v in enumerate(chunks) if i != c_axis) for chunks in md["chunksize"]]
    md["name"] = name
    md["dtype"] = dtype
    md["data_type"] = "label"
    md["channel_names"] = []
    md["channel_colors"] = []
    return md

## v0.5 / Zarr v3 vs v0.4 / Zarr v2

PyMIF supports these pairs:

| NGFF version | Zarr format | Recommended use |
|---|---:|---|
| `"0.5"` | `3` | New default |
| `"0.4"` | `2` | Legacy compatibility |

Do not mix `ngff_version="0.5"` with `zarr_format=2`, or `ngff_version="0.4"` with `zarr_format=3`.

In [ ]:
manager = make_small_raw_manager(seed=50, num_levels=2)

v05 = clean_path(OUTPUT_DIR / "legacy_demo_v05.zarr")
manager.to_zarr(v05, ngff_version="0.5", zarr_format=3)
read_v05 = mm.ZarrManager(v05, mode="r")
print("v0.5 zarr_format metadata:", read_v05.metadata.get("zarr_format"))

v04 = clean_path(OUTPUT_DIR / "legacy_demo_v04.zarr")
manager.to_zarr(v04, ngff_version="0.4", zarr_format=2)
read_v04 = mm.ZarrManager(v04, mode="r")
print("v0.4 zarr_format metadata:", read_v04.metadata.get("zarr_format"))

## `ZarrV04Manager` is a compatibility wrapper

Use it only when you want to document that a dataset is old v0.4 style. For normal code, `ZarrManager` already reads v0.4 and v0.5.

In [ ]:
legacy_wrapper = mm.ZarrV04Manager(v04)
summarize_manager(legacy_wrapper, "read with ZarrV04Manager")

## Axis-aware examples

Old examples often assumed `TCZYX`. Current PyMIF can use any unique subset of `t`, `c`, `z`, `y`, `x`.

In [ ]:
examples = [
    ("yx", (128, 128), [(0.5, 0.5)]),
    ("zyx", (8, 128, 128), [(2.0, 0.5, 0.5)]),
    ("tzyx", (3, 8, 128, 128), [(2.0, 0.5, 0.5)]),
    ("tczyx", (3, 2, 8, 128, 128), [(2.0, 0.5, 0.5)]),
]

for axes, shape, scales in examples:
    arr = np.zeros(shape, dtype=np.uint16)
    spatial_count = sum(ax in "zyx" for ax in axes)
    metadata = {
        "name": f"example_{axes}",
        "axes": axes,
        "size": [shape],
        "chunksize": [tuple(min(s, 32) for s in shape)],
        "scales": scales,
        "units": tuple("micrometer" for _ in range(spatial_count)),
        "dtype": "uint16",
        "data_type": "intensity",
    }
    if "t" in axes:
        metadata["time_increment"] = 1.0
        metadata["time_increment_unit"] = "second"
    if "c" in axes:
        c_size = shape[axes.index("c")]
        metadata["channel_names"] = [f"ch{i}" for i in range(c_size)]
        metadata["channel_colors"] = ["FFFFFF"] * c_size

    m = mm.ArrayManager(arr, metadata, chunks=metadata["chunksize"][0])
    print(axes, "->", m.data[0].shape, "scales", m.metadata["scales"])

## Label metadata: do not rely on implicit channel dropping

Do not pass raw intensity-image metadata when creating labels. Make label metadata explicit so `metadata["data_type"] = "label"`, axes, sizes and chunks all describe the label arrays.

In [ ]:
path = clean_path(OUTPUT_DIR / "troubleshooting_labels.zarr")
manager.to_zarr(path, ngff_version="0.5", zarr_format=3)
z = mm.ZarrManager(path, mode="r+")

# Safe explicit pattern:
label_md = label_metadata_from_image_metadata(z.metadata, name="nuclei", dtype="uint32")
z.create_empty_group("nuclei", label_md)
print("Created labels with axes:", z.labels["nuclei"].metadata["axes"])

# Avoid this style when z.metadata has data_type='intensity':
# z.create_empty_group("bad_labels", z.metadata)

## Common write errors

### Permission error

Open with `mode="r+"` or `mode="a"` before writing:

```python
z = mm.ZarrManager(path, mode="r+")
```

### Shape mismatch

The array you pass to `write_image_region` or `write_label_region` must match the selected region at the level you are writing.

### Group path confusion

Use short names for creation:

```python
```

Use full paths for label writes:

```python
z.write_label_region(labels, group="labels/nuclei")
```

### Chunks length mismatch

`metadata["chunksize"][0]` must have the same length as `metadata["axes"]` and `metadata["size"][0]`.

## Compression notes

Use no compressor while learning, then add compression once your workflow is stable.

```python
manager.to_zarr(path, ngff_version="0.5", zarr_format=3, compressor="blosc", compressor_level=3)
manager.to_zarr(path, ngff_version="0.4", zarr_format=2, compressor="gzip", compressor_level=3)
```

In this PyMIF version, the v3 writer supports `blosc` or no compressor. The v2 writer supports `blosc`, `gzip`, or no compressor.